# Validación cruzada de GrantMatch

El modelo final de la sesión pasada (Ridge con `alpha=10`, `candidato` = `final` en MLflow) frente al segundo del registro (la red neuronal de 16 neuronas), con la mediana como referencia, porque en la división única Ridge apenas le ganaba (0.361 contra 0.369).

Dos esquemas sobre los 169 awards con fecha, todos (entrenamiento y prueba de `pipeline_proyecto.py` juntos):

- **Temporal** (el que corresponde): GrantMatch estima el monto de convocatorias nuevas, así que cada pliegue entrena con los awards que empezaron antes y valida con los siguientes (`TimeSeriesSplit`, ordenados por `award_start_date`).
- **Al azar**, para comparar: `KFold` de 5 pliegues mezclado.

Además del MAE en log10, cada pliegue mide la métrica del negocio (ver abajo).

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

from pipeline_proyecto import CATEGORICAS, NUMERICAS, OBJETIVO, YA_LISTAS, datos, preparacion

# Todos los awards con fecha, del mas antiguo al mas reciente.
en_orden = datos.sort_values("award_start_date").reset_index(drop=True)
X = en_orden[NUMERICAS + CATEGORICAS + YA_LISTAS]
y = en_orden[OBJETIVO]
print("Awards:", len(en_orden), " desde", en_orden["award_start_date"].min().date(), "hasta", en_orden["award_start_date"].max().date())

MODELOS = {
    "ridge alpha=10 (final)": lambda: Ridge(alpha=10.0),
    "red neuronal": lambda: MLPRegressor(hidden_layer_sizes=(16,), alpha=1.0, solver="lbfgs", max_iter=2000, random_state=42),
    "linea base (mediana)": lambda: DummyRegressor(strategy="median"),
}
ESQUEMAS = {
    "temporal": TimeSeriesSplit(n_splits=5),
    "aleatorio": KFold(n_splits=5, shuffle=True, random_state=42),
}

Corte: 2020-01-01. Entrenamiento: 1988-08-15 a 2019-10-01; prueba: 2020-01-01 a 2025-08-01
Entrenamiento: (123, 21) Prueba: (46, 21)
['numericas__duracion_anios', 'numericas__institutions_count', 'categoricas__escala_financiamiento_centro', 'categoricas__escala_financiamiento_individual', 'categoricas__escala_financiamiento_proyecto', 'categoricas__funder_name_Agency for Healthcare Research and Quality', 'categoricas__funder_name_American Heart Association', 'categoricas__funder_name_Bill and Melinda Gates Foundation', 'categoricas__funder_name_Cancer Prevention and Research Institute of Texas', 'categoricas__funder_name_Gordon and Betty Moore Foundation', 'categoricas__funder_name_Institute of Education Sciences', 'categoricas__funder_name_John Templeton Foundation', 'categoricas__funder_name_National Aeronautics and Space Administration', 'categoricas__funder_name_National Institute of Justice', 'categoricas__funder_name_National Institutes of Health', 'categoricas__funder_name_Natio

## La métrica del negocio

GrantMatch le muestra a un investigador el monto estimado de una convocatoria para que decida si vale la pena postular. Los dos errores no cuestan lo mismo:

- **Subestimar** es el más caro: la convocatoria aparece como chica, el investigador la descarta y nunca vuelve a mirarla. Pierde una oportunidad que podía financiarle años de trabajo.
- **Sobreestimar** también cuesta, porque dedica tiempo a una propuesta que paga menos de lo que esperaba, pero se corrige solo: al leer las bases ve el tope real.

Lo aproximo como **el doble de caro subestimar que sobreestimar**. Calculo dos métricas:

- **MAE asimétrico (log10):** el error absoluto, multiplicado por 2 cuando el monto real supera al estimado.
- **% subestimado a menos de la mitad:** awards cuyo monto real es más del doble del estimado. Son los que el investigador descartaría por error.

In [2]:
FACTOR_SUBESTIMAR = 2.0


def mae_asimetrico(real, estimada):
    error = real - estimada  # positivo: el monto real es mayor (subestimado)
    return np.mean(np.where(error > 0, FACTOR_SUBESTIMAR * error, -error))


def subestimado_mitad(real, estimada):
    return np.mean(real - estimada > np.log10(2))


filas = []
for nombre_esquema, esquema in ESQUEMAS.items():
    for nombre_modelo, crear in MODELOS.items():
        for pliegue, (ent, val) in enumerate(esquema.split(X, y), start=1):
            pipeline = Pipeline([("preparar", clone(preparacion)), ("modelo", crear())])
            pipeline.fit(X.iloc[ent], y.iloc[ent])
            real, estimada = y.iloc[val].to_numpy(), pipeline.predict(X.iloc[val])
            filas.append({
                "esquema": nombre_esquema,
                "modelo": nombre_modelo,
                "pliegue": pliegue,
                "filas_entrenamiento": len(ent),
                "filas_validacion": len(val),
                "mae_log": round(mean_absolute_error(real, estimada), 4),
                "mae_asimetrico": round(mae_asimetrico(real, estimada), 4),
                "pct_subestimado_mitad": round(subestimado_mitad(real, estimada), 4),
            })

pliegues = pd.DataFrame(filas)
pliegues.to_csv("validacion_proyecto_pliegues.csv", index=False)
print(pliegues.pivot_table(index=["esquema", "modelo"], columns="pliegue", values="mae_log").to_string())

pliegue                                1       2       3       4       5
esquema   modelo                                                        
aleatorio linea base (mediana)    0.4648  0.4163  0.4213  0.5518  0.5264
          red neuronal            0.3406  0.2652  0.4736  0.4384  0.3441
          ridge alpha=10 (final)  0.4339  0.3172  0.3893  0.5083  0.4299
temporal  linea base (mediana)    0.5726  0.5801  0.5254  0.3180  0.4204
          red neuronal            0.7660  0.4102  0.4179  0.3351  0.3459
          ridge alpha=10 (final)  0.5966  0.4793  0.4478  0.2804  0.4053


## Resumen: promedio, desviación y cada pliegue

In [3]:
resumen = (
    pliegues.groupby(["esquema", "modelo"], sort=False)
    .agg(
        mae_log_promedio=("mae_log", "mean"),
        mae_log_desviacion=("mae_log", "std"),
        mae_log_peor=("mae_log", "max"),
        mae_asimetrico_promedio=("mae_asimetrico", "mean"),
        pct_subestimado_mitad=("pct_subestimado_mitad", "mean"),
        pliegues_mae_log=("mae_log", lambda s: list(s.round(3))),
    )
    .round(4)
    .reset_index()
)
resumen.to_csv("validacion_proyecto_resumen.csv", index=False)
print(resumen.to_string(index=False))

  esquema                 modelo  mae_log_promedio  mae_log_desviacion  mae_log_peor  mae_asimetrico_promedio  pct_subestimado_mitad                    pliegues_mae_log
 temporal ridge alpha=10 (final)            0.4419              0.1149        0.5966                   0.7213                 0.3071  [0.597, 0.479, 0.448, 0.28, 0.405]
 temporal           red neuronal            0.4550              0.1778        0.7660                   0.7531                 0.3072  [0.766, 0.41, 0.418, 0.335, 0.346]
 temporal   linea base (mediana)            0.4833              0.1122        0.5801                   0.8056                 0.3000   [0.573, 0.58, 0.525, 0.318, 0.42]
aleatorio ridge alpha=10 (final)            0.4157              0.0698        0.5083                   0.6256                 0.2604  [0.434, 0.317, 0.389, 0.508, 0.43]
aleatorio           red neuronal            0.3724              0.0835        0.4736                   0.5779                 0.2424 [0.341, 0.265, 0.474, 

## ¿La diferencia entre los modelos es mayor que la variación entre pliegues?

In [4]:
for nombre_esquema in ESQUEMAS:
    tabla = pliegues[pliegues["esquema"] == nombre_esquema].pivot(index="pliegue", columns="modelo", values="mae_log")
    diferencia = tabla["red neuronal"] - tabla["ridge alpha=10 (final)"]
    contra_base = tabla["linea base (mediana)"] - tabla["ridge alpha=10 (final)"]
    print(f"{nombre_esquema:10s} red - ridge por pliegue {list(diferencia.round(3))}  promedio {diferencia.mean():+.3f}  "
          f"(la red gana en {(diferencia < 0).sum()} de 5)")
    print(f"{'':10s} mediana - ridge por pliegue {list(contra_base.round(3))}  promedio {contra_base.mean():+.3f}  "
          f"(ridge gana en {(contra_base > 0).sum()} de 5)")

temporal   red - ridge por pliegue [0.169, -0.069, -0.03, 0.055, -0.059]  promedio +0.013  (la red gana en 3 de 5)
           mediana - ridge por pliegue [-0.024, 0.101, 0.078, 0.038, 0.015]  promedio +0.041  (ridge gana en 4 de 5)
aleatorio  red - ridge por pliegue [-0.093, -0.052, 0.084, -0.07, -0.086]  promedio -0.043  (la red gana en 4 de 5)
           mediana - ridge por pliegue [0.031, 0.099, 0.032, 0.043, 0.096]  promedio +0.060  (ridge gana en 5 de 5)
